# Document Analysis & Visualization

Load a PDF or CSV, extract structured data via LLM, render interactive Plotly charts.

**Supported Providers:** OpenAI · Anthropic · Ollama — switch with a single variable.

## Overview

| Decision | Choice | Reason |
|---|---|---|
| API keys | `.env` file + `python-dotenv` | Standard local dev pattern, no Kaggle dependency |
| Provider switching | Single `LLM_PROVIDER` variable | One-line swap, no code changes elsewhere |
| Extraction method | `with_structured_output(Pydantic)` | Guaranteed JSON, type-validated |
| Visualization | Plotly Express | Interactive in Jupyter, minimal code |
| PDF loader | `PyMuPDFLoader` | Fastest community loader |

## Step 1 — Install Dependencies
Install `PyMuPDF` (PDF extraction), `Plotly` (interactive charts), `python-dotenv` (.env support), and `LangChain` provider packages for OpenAI, Anthropic, and Ollama.

In [85]:
%pip install pymupdf plotly pandas nbformat python-dotenv \
             langchain-openai langchain-anthropic langchain-ollama \
             langchain-community
print("Step 1 done: all packages installed.")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Step 1 done: all packages installed.


## Step 2 — Provider Configuration
Set `LLM_PROVIDER` (one of `openai`, `anthropic`, or `ollama`). API keys are loaded from the environment or a `.env` file in the project root.

In [86]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env file if present

# ── Choose your provider ──────────────────────────────
LLM_PROVIDER = "ollama"   # "openai" | "anthropic" | "ollama"

# API keys — set here or in a .env file
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "sk-...")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-...")
# Ollama needs no API key — just run: `ollama pull llama3.2`

# Models — override via .env or set here directly
OPENAI_MODEL    = os.getenv("OPENAI_MODEL",    "gpt-4o-mini")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL",    "llama3.2")

print(f"✅: provider = '{LLM_PROVIDER}'.")
print(f"    models: openai={OPENAI_MODEL}, anthropic={ANTHROPIC_MODEL}, ollama={OLLAMA_MODEL}")

✅: provider = 'ollama'.
    models: openai=gpt-4o-mini, anthropic=claude-sonnet-4-6, ollama=llama3.2


## Step 3 — LLM Factory
`build_llm()` returns the correct LangChain chat model for the chosen provider. Only the selected provider's package is imported, so unused providers do not need to be installed.

In [87]:
def build_llm(provider: str):
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0)

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
        return ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)

    elif provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=OLLAMA_MODEL, temperature=0)

    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'openai', 'anthropic', or 'ollama'.")

llm = build_llm(LLM_PROVIDER)
_active_model = {"openai": OPENAI_MODEL, "anthropic": ANTHROPIC_MODEL, "ollama": OLLAMA_MODEL}[LLM_PROVIDER]
print(f"✅: {type(llm).__name__} initialised with model '{_active_model}'.")

✅: ChatOllama initialised with model 'llama3.2'.


## Step 4 — Imports
Load all runtime dependencies: pandas for tabular data, Plotly for charts, Pydantic for the extraction schema, and LangChain document loaders for PDF and CSV ingestion.

In [88]:
from pathlib import Path
from typing import List, Optional

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_core.prompts import ChatPromptTemplate

print("✅: all imports loaded.")

✅: all imports loaded.


## Step 5 — Extraction Schema (Pydantic)
`DataPoint` represents one extracted number with its label, category, and optional unit. `ExtractedReport` is the top-level container the LLM must populate: title, a 2–3 sentence summary, a list of `DataPoint`s, and a key-metrics dict.

In [89]:
class DataPoint(BaseModel):
    label: str = Field(description="Name or label of the data point")
    value: float = Field(description="Numeric value")
    category: str = Field(description="Category or group this belongs to")
    unit: Optional[str] = Field(default=None, description="Unit of measurement if present")

class ExtractedReport(BaseModel):
    title: str = Field(description="Title or topic of the document")
    summary: str = Field(description="2-3 sentence summary of key findings")
    data_points: List[DataPoint] = Field(description="All numerical data found")
    key_metrics: dict = Field(description="Top-level KPIs as name:value pairs")
print("✅: DataPoint and ExtractedReport schemas defined.")

✅: DataPoint and ExtractedReport schemas defined.


## Step 6 — Document Loader
`load_document()` detects file type by extension: PyMuPDFLoader for `.pdf` and CSVLoader for `.csv`. Returns a single concatenated string ready for the LLM prompt.

In [90]:
def load_document(file_path: str) -> str:
    path = Path(file_path)
    if path.suffix.lower() == ".pdf":
        docs = PyMuPDFLoader(file_path).load()
        return "\n\n".join(d.page_content for d in docs)
    elif path.suffix.lower() == ".csv":
        docs = CSVLoader(file_path).load()
        return "\n".join(d.page_content for d in docs)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}. Use .pdf or .csv")
print("✅: load_document() ready.")

✅: load_document() ready.


## Step 7 — Extraction Chain
Bind the LLM to the Pydantic schema via `with_structured_output()`, then compose an LCEL chain: prompt | structured LLM. The chain guarantees type-validated, schema-conforming JSON on every call.

In [91]:
structured_llm = llm.with_structured_output(ExtractedReport)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction specialist.
Analyze the document and extract ALL numerical data points, metrics, and statistics.
Be precise with values. Capture units when present."""),
    ("human", "Document content:\n\n{content}\n\nExtract all structured data.")
])

extraction_chain = prompt | structured_llm
print("✅: extraction_chain ready.")

✅: extraction_chain ready.


## Step 8 — Load Document & Run Extraction
Set `FILE_PATH` to your PDF or CSV, load its text, and invoke the extraction chain. The LLM returns a fully populated `ExtractedReport` with title, summary, data points, and key metrics.

In [ ]:
FILE_PATH = "sample_report.csv"   # <- change to your file

content = load_document(FILE_PATH)
print(f"Loaded {len(content):,} characters from {FILE_PATH}")

result: ExtractedReport = extraction_chain.invoke({"content": content})

print(f"\nTitle:        {result.title}")
print(f"Summary:      {result.summary}")
print(f"Data points:  {len(result.data_points)}")
print(f"Key metrics:  {list(result.key_metrics.keys())}")
print(f"\n✅: {len(result.data_points)} data points and {len(result.key_metrics)} key metrics extracted.")

Loaded 1,532 characters from sample_report.csv


## Step 9 — Build DataFrame
Flatten the `data_points` list into a pandas DataFrame with columns `label`, `value`, `category`, and `unit`. This DataFrame is the shared data source for all three visualisation cells below.

In [ ]:
df = pd.DataFrame([
    {"label": dp.label, "value": dp.value, "category": dp.category, "unit": dp.unit}
    for dp in result.data_points
])
print(df.to_string(index=False))
print(f"\n✅: {len(df)} rows, {df['category'].nunique()} unique categories.")

## Step 10 — Bar Chart (Values by Label)
Grouped bar chart showing every extracted value coloured by category. Labels are rotated 40 degrees for readability; hover over any bar for the exact number.

In [ ]:
fig_bar = px.bar(
    df, x="label", y="value", color="category",
    title=f"{result.title} — Values by Label",
    text_auto=True, height=450
)
fig_bar.update_layout(xaxis_tickangle=-40)
display(HTML(fig_bar.to_html(include_plotlyjs="cdn", full_html=False)))
print(f"✅: bar chart with {len(df)} bars rendered.")

## Step 11 — Pie Chart (Category Totals)
Donut chart aggregating total values per category. Gives an instant proportional breakdown of which category dominates the document.

In [ ]:
cat_totals = df.groupby("category")["value"].sum().reset_index()

fig_pie = px.pie(
    cat_totals, values="value", names="category",
    title=f"{result.title} — Distribution by Category",
    hole=0.35
)
display(HTML(fig_pie.to_html(include_plotlyjs="cdn", full_html=False)))
print(f"✅: pie chart with {len(cat_totals)} slices rendered.")

## Step 12 — Key Metrics Table
Interactive Plotly table displaying the high-level KPIs the LLM identified. The blue header row visually separates metric names from their values.

In [ ]:
metrics_df = pd.DataFrame(
    [(k, str(v)) for k, v in result.key_metrics.items()],
    columns=["Metric", "Value"]
)

fig_tbl = go.Figure(data=[go.Table(
    header=dict(
        values=["Metric", "Value"],
        fill_color="#4C78A8",
        font=dict(color="white", size=13)
    ),
    cells=dict(values=[metrics_df["Metric"], metrics_df["Value"]])
)])
fig_tbl.update_layout(title="Key Metrics Summary")
display(HTML(fig_tbl.to_html(include_plotlyjs="cdn", full_html=False)))
print(f"✅: metrics table with {len(metrics_df)} rows rendered.")

## Step 13 — Final Summary
Print a concise human-readable recap: document title, key findings, the list of categories discovered, and total data point count.

In [ ]:
print("=" * 60)
print(f"  {result.title}")
print("=" * 60)
print(f"\n{result.summary}")
print(f"\nCategories : {df['category'].unique().tolist()}")
print(f"Data points: {len(df)}")
print("\n✅ Analysis complete.")

In [ ]:
from pathlib import Path

def _build_html_report():
    bar_html = fig_bar.to_html(include_plotlyjs=True,  full_html=False)
    pie_html = fig_pie.to_html(include_plotlyjs=False, full_html=False)
    tbl_html = fig_tbl.to_html(include_plotlyjs=False, full_html=False)

    metrics_rows = "".join(
        f"<tr><td>{k}</td><td>{v}</td></tr>"
        for k, v in result.key_metrics.items()
    )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{result.title}</title>
  <style>
    body {{ font-family: sans-serif; max-width: 1100px; margin: 40px auto; padding: 0 20px; color: #2a3f5f; }}
    h1 {{ border-bottom: 2px solid #4C78A8; padding-bottom: 8px; }}
    h2 {{ margin-top: 40px; color: #4C78A8; }}
    .meta {{ color: #888; font-size: 0.9em; margin-bottom: 24px; }}
    table {{ border-collapse: collapse; width: 100%; margin-top: 12px; }}
    th {{ background: #4C78A8; color: white; padding: 8px 12px; text-align: left; }}
    td {{ padding: 7px 12px; border-bottom: 1px solid #e0e6f0; }}
    tr:hover td {{ background: #f4f7fb; }}
  </style>
</head>
<body>
  <h1>{result.title}</h1>
  <p class="meta">Source: <code>{FILE_PATH}</code> &nbsp;|&nbsp;
     Provider: <code>{LLM_PROVIDER}</code> &nbsp;|&nbsp;
     Model: <code>{_active_model}</code></p>

  <h2>Summary</h2>
  <p>{result.summary}</p>

  <h2>Key Metrics</h2>
  <table>
    <tr><th>Metric</th><th>Value</th></tr>
    {metrics_rows}
  </table>

  <h2>Bar Chart — ValuesK
  {bar_html}

  <h2>Pie Chart — Distribution by Category</h2>
  {pie_html}

  <h2>Key Metrics Table</h2>
  {tbl_html}
</body>
</html>"""

out_path = Path(FILE_PATH).with_suffix(".html")
out_path.write_text(_build_html_report(), encoding="utf-8")
print(f"\u2705: report saved to \'{out_path}\' ({out_path.stat().st_size:,} bytes)")
K

## Step 14 — Export to HTML
Build a self-contained HTML report that embeds Plotly.js and all three charts inline — no CDN, no internet connection required to open it.

The file is written next to the notebook.